# RQ1: Does personality impact on testing?

This notebook computes the healthiness of the 120 runs of the `event x personality` grid (15 events x 8 personality populations) and the **HCS failure verdict** of each run, then derives the RQ1 results of the paper: the verdict table, the failure counts and timings, the severity of the failing spikes, the comparison between the *general* population and the personality populations, the comparison with the shallow personalities of prior work, and the test suites obtained with and without personalities.

**Healthiness index.** For every run we compute the *failed requests* index

$$ \mathrm{FR}(t) = \frac{\texttt{rides\_not\_served}(t) + \texttt{passengers\_cancel}(t)}{\texttt{passengers\_new}(t)} \cdot \min\big(1,\ a\ln(\texttt{passengers\_new}(t)+b)+c\big) $$

on the 600 s rolling mean of the run, after discarding the first three hours (warm-up) and the last two hours (tail). 
The demand weight is fitted on the threshold baseline so that it is 0 at zero demand and 1 at the baseline's mean demand.

**Healthiness dimensions and thresholds from Puccinelli et al..** *Hardiness* is the level of $\mathrm{FR}(t)$, *consistency* its 600 s rolling standard deviation, *resilience* the time $\mathrm{FR}(t)$ stays above its own trend. 
The three thresholds are the 99th percentiles measured on the `normal` event with the *general* population (`datasets/raw/normal/ocean/sf_normal_ocean_dr_general_pass_general.csv`).
A run fails (HCS failure) when the three dimensions are simultaneously out of range.

In [1]:
import copy
import glob
import json
import os
import pickle
import re
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

In [2]:
%run utils.ipynb

In [3]:
# overwrite = True recomputes every run from the raw CSVs; False reuses the per-run pickles
overwrite = True
# The per-run plots are always saved to disk; set to True to also display them here
SHOW_RUN_PLOTS = False
AGGREGATE = 600
WINDOW_FILL = 60
CUT = 7200

dataRoot = os.path.join("datasets", "raw")
outRoot = os.path.join("analysis_out", "rq1-healthiness")
cacheDir = os.path.join(outRoot, "runs")
plotsRoot = os.path.join(outRoot, "plots")
resultsPath = os.path.join(outRoot, "healthiness_results.csv")
os.makedirs(cacheDir, exist_ok=True)
if overwrite:
    for file in glob.glob(os.path.join(cacheDir, "*.pkl")):
        os.remove(file)

analysis_start = 3600 + CUT - AGGREGATE
# Absolute timestamp at which the event and the personality are injected (19:00 of day two)
event_injection = 79200

## 1. Per-run healthiness

In [4]:
# Thresholds and scaling parameters depend on the baseline run, so they are computed once per baseline and cached
def compute_baseline_thresholds(base_name, training_path):
    baseline_file = os.path.join(cacheDir, f"baseline_{base_name}.pkl")
    if os.path.exists(baseline_file):
        with open(baseline_file, 'rb') as f:
            return pickle.load(f)
    df = pd.read_csv(training_path)
    baseline = compute_thresholds(df, AGGREGATE, CUT)
    with open(baseline_file, 'wb') as f:
        pickle.dump(baseline, f)
    perc_99_h, perc_99_c, perc_99_r, a, b, c = baseline
    print(f"  hardiness 99th perc.: {round(perc_99_h, 3)}, "
          f"consistency 99th perc.: {round(perc_99_c, 3)}, "
          f"resilience 99th perc.: {perc_99_r} timestamps")
    return baseline

In [5]:
def run_pickle_path(scenario, driver_job, passenger_job, base_name):
    return os.path.join(cacheDir, f"thresholds_{scenario}_{driver_job}_{passenger_job}_{base_name}.pkl")


# Compute the healthiness metrics of one event run against one baseline
def compute_metrics(baseline, event_path, scenario_info):
    # Check if results already exist
    thresholds_file = run_pickle_path(*scenario_info)
    if os.path.exists(thresholds_file):
        with open(thresholds_file, 'rb') as f:
            return pickle.load(f)

    perc_99_h, perc_99_c, perc_99_r, a, b, c = baseline
    df_event = pd.read_csv(event_path)
    df_event = fill_missing_data(df_event, WINDOW_FILL)
    df_failed_requests_event = copy.deepcopy(df_event)
    (hardiness_event, exceed_timestamp_h, exceed_timestamp_c, exceed_timestamp_r, consecutive_t) = compute_exceeding_timestamps(df_failed_requests_event, AGGREGATE, CUT, a, b, c, perc_99_h, perc_99_c, perc_99_r)
    failure_timestamp = find_hcs_failure(exceed_timestamp_h, exceed_timestamp_c, exceed_timestamp_r, consecutive_t, perc_99_r)
    if failure_timestamp > 0:
        print(f"  HCS failure at timestamp: {failure_timestamp + analysis_start}")
    exceed_timestamp_h = get_first_above_threshold(exceed_timestamp_h, 600)
    exceed_timestamp_c = get_first_above_threshold(exceed_timestamp_c, 600)

    # Save pickle file with the metrics of this event
    thresholds = (
        hardiness_event,
        exceed_timestamp_h,
        exceed_timestamp_c,
        exceed_timestamp_r,
        perc_99_h,
        perc_99_c,
        perc_99_r,
        consecutive_t,
        failure_timestamp
        )
    with open(thresholds_file, 'wb') as f:
        pickle.dump(thresholds, f)

    return thresholds

In [6]:
# Plot the hardiness metric of an event run against the baseline thresholds
def plot_hardiness(title, plots_dir, filename, hardiness_event, exceed_timestamp_h, exceed_timestamp_c, exceed_timestamp_r, perc_99_h, perc_99_c, perc_99_r, consecutive_t, failure_timestamp):
    start_timestamp = analysis_start
    end_timestamp = start_timestamp + len(hardiness_event)
    timestamps_x = np.arange(start_timestamp, start_timestamp + len(hardiness_event))
    os.makedirs(plots_dir, exist_ok=True)
    with plt.rc_context({"figure.figsize": (15, 10), "font.size": 24}):
        plt.figure()
        if failure_timestamp > 0:
            plt.plot(timestamps_x[:failure_timestamp], hardiness_event[:failure_timestamp], label='Failed requests')
        else:
            plt.plot(timestamps_x, hardiness_event, label='Failed requests')
        for exceed_r, run_length in zip(exceed_timestamp_r, consecutive_t):
            start_idx = max(0, exceed_r - perc_99_r)
            end_idx = exceed_r + run_length - perc_99_r
            if failure_timestamp > 0:
                end_idx = min(failure_timestamp, end_idx)
            if start_idx < end_idx:
                plt.plot(timestamps_x[start_idx:end_idx], hardiness_event[start_idx:end_idx], color='purple')
        plt.title(title)
        plt.xlabel('Timestamp')
        plt.ylabel('Percentage (x100%)')
        plt.axhline(y=perc_99_h, color='orange', linestyle='--', label='Hardiness 99th perc.')
        plt.axvline(x=event_injection, color='blue', label='Event injection')
        handles, labels = plt.gca().get_legend_handles_labels()
        already_labeled = set(labels)
        for el in exceed_timestamp_h:
            if failure_timestamp <= 0 or el <= failure_timestamp:
                lbl = 'Hardiness OOR' if 'Hardiness OOR' not in already_labeled else ""
                plt.axvline(x=el + start_timestamp, color='red', label=lbl)
                already_labeled.add('Hardiness OOR')
        for el in exceed_timestamp_c:
            if failure_timestamp <= 0 or el <= failure_timestamp:
                lbl = 'Consistency OOR' if 'Consistency OOR' not in already_labeled else ""
                plt.axvline(x=el + start_timestamp, color='orange', label=lbl)
                already_labeled.add('Consistency OOR')
        for exceed_r in exceed_timestamp_r:
            if failure_timestamp <= 0 or exceed_r <= failure_timestamp:
                lbl = 'Resilience OOR' if 'Resilience OOR' not in already_labeled else ""
                plt.axvline(x=exceed_r + start_timestamp, color='purple', label=lbl)
                already_labeled.add('Resilience OOR')
        plt.ylim(0, 1)
        plt.xlim(start_timestamp, end_timestamp)
        plt.legend(loc='upper left')
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(os.path.join(plots_dir, filename), dpi=300)
        if SHOW_RUN_PLOTS:
            plt.show()
        plt.close()

## 2. Healthiness and verdict of every run

In [7]:
# Thresholds are fitted on the no-personality run (general driver, general passenger)
base_name = "sf_normal_ocean"
training_path = os.path.join(dataRoot, "normal/ocean/sf_normal_ocean_dr_general_pass_general.csv")
if not os.path.exists(training_path):
    raise FileNotFoundError(f"baseline run not found at {training_path}")
ocean_event_paths = sorted(glob.glob(os.path.join(dataRoot, "*/ocean/*_dr_*_pass_*.csv")))
# Files are named sf_<event>_ocean_dr_<driver job>_pass_<passenger job>.csv
RUN_FILE_RE = re.compile(r"_dr_(?P<driver>.+?)_pass_(?P<passenger>.+)$")
print(f"Found {len(ocean_event_paths)} runs")
baseline = compute_baseline_thresholds(base_name, training_path)
perc_99_h, perc_99_c, perc_99_r, a, b, c = baseline
with open(os.path.join(outRoot, "thresholds.json"), "w") as f:
    json.dump({"baseline": base_name, "p99_h": float(perc_99_h), "p99_c": float(perc_99_c),
               "p99_r": int(perc_99_r), "a": float(a), "b": float(b), "c": float(c),
               "aggregate": AGGREGATE, "window_fill": WINDOW_FILL, "cut": CUT,
               "analysis_start": analysis_start, "event_injection": event_injection}, f, indent=2)

results = []
for event_path in ocean_event_paths:
    scenario = event_path.split(os.sep)[-3]
    m = RUN_FILE_RE.search(os.path.splitext(os.path.basename(event_path))[0])
    if m is None:
        print(f"WARNING: cannot parse the run identity of {event_path}.")
        continue
    driver_job, passenger_job = m.group("driver").lower(), m.group("passenger").lower()
    plot_title = f"Indexes of healthiness for event {scenario}\ndriver: {driver_job}, passenger: {passenger_job}, baseline: {base_name}"
    plots_dir = os.path.join(plotsRoot, scenario)
    filename = f"healthiness_{driver_job}_{passenger_job}.pdf"
    print(f"Processing event: {scenario}, driver job: {driver_job}, passenger job: {passenger_job}")
    thresholds = compute_metrics(baseline, event_path, (scenario, driver_job, passenger_job, base_name))
    failure_timestamp = thresholds[-1]
    results.append({
        "scenario": scenario,
        "driver_job": driver_job,
        "passenger_job": passenger_job,
        "baseline": base_name,
        "failure": failure_timestamp > 0,
        "failure_timestamp": failure_timestamp + analysis_start if failure_timestamp > 0 else None
    })
    if not overwrite and os.path.exists(os.path.join(plots_dir, filename)):
        continue
    plot_hardiness(plot_title, plots_dir, filename, *thresholds)

results_df = pd.DataFrame(results, columns=["scenario", "driver_job", "passenger_job", "baseline", "failure", "failure_timestamp"])
if results_df.empty:
    raise RuntimeError("No run was processed.")
results_df.to_csv(resultsPath, index=False)
print(f"\n{len(results_df)} runs, {int(results_df.failure.sum())} HCS failures -> {resultsPath}")

Found 120 runs


  hardiness 99th perc.: 0.223, consistency 99th perc.: 0.028, resilience 99th perc.: 2894 timestamps
Processing event: boycott_tncs-long_rides, driver job: driver, passenger job: bank_tellers


  HCS failure at timestamp: 86461


Processing event: boycott_tncs-long_rides, driver job: driver, passenger job: general


Processing event: boycott_tncs-long_rides, driver job: driver, passenger job: managers


...

Processing event: wildcat_strike, driver job: general, passenger job: bank_tellers


Processing event: wildcat_strike, driver job: general, passenger job: general


Processing event: wildcat_strike, driver job: general, passenger job: managers


Processing event: wildcat_strike, driver job: general, passenger job: students



120 runs, 57 HCS failures -> analysis_out/rq1-healthiness/healthiness_results.csv


## 3. RQ1 results

### Verdicts table

In [8]:
EVENTS = {"underground_alarm": "UA", "wildcat_strike": "WS", "flash_mob": "FM", "long_rides": "LR", "boycott_tncs": "BT"}
SCENARIO_ORDER = ["normal", "underground_alarm", "wildcat_strike", "flash_mob", "long_rides", "boycott_tncs", 
                  "underground_alarm-flash_mob", "underground_alarm-long_rides", "underground_alarm-wildcat_strike", 
                  "flash_mob-long_rides", "flash_mob-wildcat_strike", "flash_mob-boycott_tncs", 
                  "wildcat_strike-long_rides", "wildcat_strike-boycott_tncs", "boycott_tncs-long_rides"]
POPULATIONS = ["baseline", "students", "managers", "bank_tellers", "drivers", "students_drivers", "managers_drivers", "bank_tellers_drivers"]
POPULATION_LABEL = {"baseline": "general", "students": "students", "managers": "managers", "bank_tellers": "bank tellers", "drivers": "drivers", 
                    "students_drivers": "students + drivers", "managers_drivers": "managers + drivers", "bank_tellers_drivers": "bank tellers + drivers"}
DRIVER_JOB = {"driver": "driver", "general": "general"}

def personality_of(driver_job: str, passenger_job: str) -> str:
    has_driver = driver_job == "driver"
    if passenger_job == "general":
        return "drivers" if has_driver else "baseline"
    return f"{passenger_job}_drivers" if has_driver else passenger_job

def scenario_label(scenario: str) -> str:
    return "Normal" if scenario == "normal" else " + ".join(EVENTS[e] for e in scenario.split("-"))

def has_drivers(population: str) -> bool:
    return population == "drivers" or population.endswith("_drivers")

def has_passenger_personality(population: str) -> bool:
    return population not in ("baseline", "drivers")

verdicts = results_df.copy()
verdicts["personality"] = [personality_of(d, p) for d, p in zip(verdicts.driver_job, verdicts.passenger_job)]
verdicts["is_compound"] = verdicts.scenario.str.contains("-")
verdicts["minutes"] = (verdicts.failure_timestamp - event_injection) / 60.0
missing = set(SCENARIO_ORDER) - set(verdicts.scenario)
assert not missing, f"events missing from datasets/raw: {missing}"
assert len(verdicts) == len(SCENARIO_ORDER) * len(POPULATIONS), "the grid is incomplete"
EVENT_SCENARIOS = [s for s in SCENARIO_ORDER if s != "normal"]
summary = {}

### Failure counts and timing

In [9]:
fail = verdicts[verdicts.failure]
single = verdicts[(verdicts.scenario != "normal") & ~verdicts.is_compound]
compound = verdicts[verdicts.is_compound]
summary["runs"] = int(len(verdicts))
summary["failures"] = int(verdicts.failure.sum())
summary["single_failures"] = int(single.failure.sum())
summary["compound_failures"] = int(compound.failure.sum())
summary["minutes_to_failure"] = {"median": round(float(fail.minutes.median()), 1),
                                 "q1": round(float(fail.minutes.quantile(.25)), 1),
                                 "q3": round(float(fail.minutes.quantile(.75)), 1),
                                 "min": round(float(fail.minutes.min()), 1),
                                 "max": round(float(fail.minutes.max()), 1)}
for k in ("runs", "failures", "single_failures", "compound_failures", "minutes_to_failure"):
    print(f"{k:>20}: {summary[k]}")

                runs: 120
            failures: 57
     single_failures: 14
   compound_failures: 43
  minutes_to_failure: {'median': 59.2, 'q1': 54.0, 'q3': 63.5, 'min': 5.2, 'max': 147.8}


In [10]:
STRIDE = 20
NORMAL_WINDOW = (event_injection, event_injection + 1800)

def load_payload(scenario, driver_job, passenger_job):
    with open(run_pickle_path(scenario, driver_job, passenger_job, base_name), "rb") as f:
        return pickle.load(f)

rows = []
for r in verdicts[verdicts.scenario == "normal"].itertuples():
    x = np.asarray(load_payload(r.scenario, r.driver_job, r.passenger_job)[0], dtype=float)[::STRIDE]
    ts = analysis_start + STRIDE * np.arange(len(x))
    inside = (ts >= NORMAL_WINDOW[0]) & (ts < NORMAL_WINDOW[1])
    rows.append({"personality": r.personality, "fr_mean_window": float(x[inside].mean())})
normal_levels = pd.DataFrame(rows).set_index("personality").reindex(POPULATIONS)
general_level = float(normal_levels.loc["baseline", "fr_mean_window"])
normal_levels["diff_vs_general"] = normal_levels.fr_mean_window - general_level
normal_levels.to_csv(os.path.join(outRoot, "normal_levels.csv"))
summary["normal_general_level"] = round(general_level, 3)
summary["normal_max_abs_diff"] = round(float(normal_levels.diff_vs_general.abs().max()), 3)
display(normal_levels.round(4))

,fr_mean_window,diff_vs_general
personality,,
baseline,0.1769,0.0000
students,0.1745,-0.0024
managers,0.1814,0.0045
bank_tellers,0.1676,-0.0094
drivers,0.1639,-0.0130
students_drivers,0.1637,-0.0132
managers_drivers,0.1821,0.0052
bank_tellers_drivers,0.1834,0.0065


### Severity of the failing spikes

In [11]:
SPIKE_GAP_S = 600
MIN_SPIKE_S = 300

def find_spikes(ts, x, threshold, gap_s=SPIKE_GAP_S, min_duration_s=MIN_SPIKE_S):
    ts, x = np.asarray(ts, dtype=float), np.asarray(x, dtype=float)
    idx = np.flatnonzero(np.isfinite(x) & (x > threshold))
    if idx.size == 0:
        return []
    breaks = np.flatnonzero(np.diff(ts[idx]) > gap_s)
    starts = idx[np.insert(breaks + 1, 0, 0)]
    ends = idx[np.append(breaks, len(idx) - 1)]
    return [(float(ts[s]), float(ts[e] + STRIDE)) for s, e in zip(starts, ends)
            if ts[e] - ts[s] + STRIDE >= min_duration_s]

rows = []
for r in fail.itertuples():
    (hardiness, _, _, exceed_r, p99_h, p99_c, p99_r, consecutive_t, failure_ts) = load_payload(
        r.scenario, r.driver_job, r.passenger_job)
    idx = np.asarray(hardiness, dtype=float)
    consistency = pd.Series(idx).rolling(window=AGGREGATE).std().fillna(0).to_numpy()
    ts = analysis_start + np.arange(len(idx))
    full_h, full_c = np.flatnonzero(idx > p99_h), np.flatnonzero(consistency > p99_c)
    run_ticks = np.nan
    for er, run_len in zip(exceed_r, consecutive_t):
        w0, w1 = er - p99_r, er - p99_r + run_len
        if ((full_h >= w0) & (full_h <= w1)).any() and ((full_c >= w0) & (full_c <= w1)).any():
            run_ticks = float(run_len)
            break
    fail_abs = failure_ts + analysis_start
    spike = next((sp for sp in find_spikes(ts[::STRIDE], idx[::STRIDE], p99_h)
                  if sp[0] <= fail_abs < sp[1]), None)
    if spike is None:
        continue
    inside = (ts >= spike[0]) & (ts < spike[1])
    rows.append({"scenario": r.scenario, "personality": r.personality,
                 "minutes": r.minutes, "spike_start": spike[0], "spike_end": spike[1],
                 "hardiness_peak": float(idx[inside].max()),
                 "consistency_peak": float(consistency[inside].max()),
                 "resilience_ticks": run_ticks})
severity = pd.DataFrame(rows)
severity.to_csv(os.path.join(outRoot, "failure_severity.csv"), index=False)

population = (verdicts.groupby("personality").failure.sum().rename("failures").to_frame()
              .join(severity.groupby("personality")[["hardiness_peak", "consistency_peak", "resilience_ticks"]].mean())
              .reindex(POPULATIONS))
population.index = [POPULATION_LABEL[p] for p in population.index]
population.to_csv(os.path.join(outRoot, "population_summary.csv"))
display(population.round(3))

,failures,hardiness_peak,consistency_peak,resilience_ticks
general,4,0.613,0.053,7790.750
students,6,0.572,0.063,6370.667
managers,7,0.571,0.066,6214.286
bank tellers,7,0.575,0.051,7431.333
drivers,6,0.579,0.062,6209.000
students + drivers,7,0.551,0.056,6704.714
managers + drivers,8,0.528,0.052,5967.250
bank tellers + drivers,12,0.477,0.045,6282.000


### General population versus personalities

In [12]:
# Minutes from the injection to the HCS failure, one row per event and one column per population
minutes = (verdicts.pivot(index="scenario", columns="personality", values="minutes")
           .reindex(index=SCENARIO_ORDER, columns=POPULATIONS))
failing_by = {s: [p for p in POPULATIONS if not pd.isna(minutes.at[s, p])] for s in EVENT_SCENARIOS}
suite = pd.DataFrame([{
    "scenario": s, "label": scenario_label(s),
    "general_fails": "baseline" in failing_by[s],
    "failing_populations": len(failing_by[s]),
    "populations": ", ".join(POPULATION_LABEL[p] for p in failing_by[s]),
    "only_professional_drivers": bool(failing_by[s]) and "baseline" not in failing_by[s] and all(has_drivers(p) for p in failing_by[s]),
    "only_passenger_personalities": bool(failing_by[s]) and "baseline" not in failing_by[s] and all(has_passenger_personality(p) for p in failing_by[s]),
} for s in EVENT_SCENARIOS])
suite.to_csv(os.path.join(outRoot, "test_suite.csv"), index=False)
display(suite.rename(columns={"scenario": "event"}))

general_passes = [s for s in EVENT_SCENARIOS if "baseline" not in failing_by[s]]
hidden = verdicts[verdicts.scenario.isin(general_passes) & (verdicts.personality != "baseline")]
summary["general_failures"] = int(suite.general_fails.sum())
summary["failures_per_population"] = {POPULATION_LABEL[p]: int(minutes[p].notna().sum()) for p in POPULATIONS}
summary["general_passes_scenarios"] = len(general_passes)
summary["failures_hidden_without_personalities"] = int(hidden.failure.sum())
summary["scenarios_exposed_with_personalities"] = int((suite.failing_populations > 0).sum())
summary["scenarios_only_professional_drivers"] = suite[suite.only_professional_drivers].label.tolist()
summary["scenarios_only_passenger_personalities"] = suite[suite.only_passenger_personalities].label.tolist()
for k in ("general_failures", "failures_per_population", "general_passes_scenarios",
          "failures_hidden_without_personalities", "scenarios_exposed_with_personalities",
          "scenarios_only_professional_drivers", "scenarios_only_passenger_personalities"):
    print(f"{k}: {summary[k]}")

,event,label,general_fails,failing_populations,populations,only_professional_drivers,only_passenger_personalities
0,underground_alarm,UA,False,7,"students, managers, bank tellers, drivers, stu...",False,False
1,wildcat_strike,WS,False,0,,False,False
2,flash_mob,FM,False,1,bank tellers + drivers,True,True
3,long_rides,LR,False,4,"drivers, students + drivers, managers + driver...",True,False
4,boycott_tncs,BT,False,2,"drivers, bank tellers + drivers",True,False
5,underground_alarm-flash_mob,UA + FM,True,8,"general, students, managers, bank tellers, dri...",False,False
6,underground_alarm-long_rides,UA + LR,True,8,"general, students, managers, bank tellers, dri...",False,False
7,underground_alarm-wildcat_strike,UA + WS,True,8,"general, students, managers, bank tellers, dri...",False,False
8,flash_mob-long_rides,FM + LR,False,3,"bank tellers, managers + drivers, bank tellers...",False,True
9,flash_mob-wildcat_strike,FM + WS,True,4,"general, managers, bank tellers, bank tellers ...",False,False


general_failures: 4
failures_per_population: {'general': 4, 'students': 6, 'managers': 7, 'bank tellers': 7, 'drivers': 6, 'students + drivers': 7, 'managers + drivers': 8, 'bank tellers + drivers': 12}
general_passes_scenarios: 10
failures_hidden_without_personalities: 29
scenarios_exposed_with_personalities: 13
scenarios_only_professional_drivers: ['FM', 'LR', 'BT', 'WS + LR', 'BT + LR']
scenarios_only_passenger_personalities: ['FM', 'FM + LR', 'FM + BT', 'WS + LR', 'WS + BT', 'BT + LR']


### Comparison with shallow personalities

In [13]:
PRIOR_TABLE = {
    "normal":                       (False, ""), 
    "underground_alarm":            (False, "hardiness and consistency out of range"),
    "flash_mob":                    (False, "hardiness and consistency out of range"),
    "wildcat_strike":               (False, "hardiness and consistency out of range"),
    "long_rides":                   (False, ""),
    "boycott_tncs":                 (False, ""),
    "flash_mob-long_rides":         (True, ""),
    "underground_alarm-long_rides": (True, ""),
    "wildcat_strike-long_rides":    (True, ""),
    "flash_mob-boycott_tncs":       (True, ""),
    "wildcat_strike-boycott_tncs":  (True, ""),
    "greedy_drivers":                   (False, "consistency out of range"),
    "budget_passengers":                (False, ""),
    "greedy_drivers-underground_alarm": (True, ""),
    "greedy_drivers-flash_mob":         (True, ""),
    "greedy_drivers-wildcat_strike":    (True, ""),
    "greedy_drivers-long_rides":        (True, ""),
    "budget_passengers-greedy_drivers": (False, "hardiness and consistency out of range"),
    "budget_passengers-wildcat_strike": (False, "hardiness and consistency out of range"),
}
PRIOR_ONLY_SINGLE = ["greedy_drivers", "budget_passengers"]
PRIOR_ONLY_COMPOUND = ["greedy_drivers-underground_alarm", "greedy_drivers-flash_mob",
                       "greedy_drivers-wildcat_strike", "greedy_drivers-long_rides",
                       "budget_passengers-greedy_drivers", "budget_passengers-wildcat_strike"]
prior = pd.DataFrame([{
    "scenario": s, "in_prior": s in PRIOR_TABLE,
    "failure": PRIOR_TABLE[s][0] if s in PRIOR_TABLE else None,
    "note": PRIOR_TABLE[s][1] if s in PRIOR_TABLE else "",
    "prior_only": s in PRIOR_ONLY_SINGLE + PRIOR_ONLY_COMPOUND,
    "prior_only_group": ("single" if s in PRIOR_ONLY_SINGLE else "compound" if s in PRIOR_ONLY_COMPOUND else ""),
} for s in SCENARIO_ORDER + PRIOR_ONLY_SINGLE + PRIOR_ONLY_COMPOUND])
prior.to_csv(os.path.join(outRoot, "prior_verdicts.csv"), index=False)

comparison = pd.DataFrame([{
    "scenario": s, "label": scenario_label(s),
    "prior_failure": PRIOR_TABLE[s][0],
    "general_failure": "baseline" in failing_by.get(s, []),
    "populations_failing": ", ".join(POPULATION_LABEL[p] for p in failing_by.get(s, [])),
} for s in EVENT_SCENARIOS if s in PRIOR_TABLE])
comparison.to_csv(os.path.join(outRoot, "prior_comparison.csv"), index=False)
display(comparison.rename(columns={"scenario": "event"}))

prior_fail = comparison[comparison.prior_failure]
summary["prior_failing_scenarios"] = prior_fail.label.tolist()
summary["prior_failures_reproduced_by_general"] = prior_fail[prior_fail.general_failure].label.tolist()
summary["single_event_verdicts_agree"] = bool(
    (comparison[~comparison.scenario.str.contains("-")].prior_failure
     == comparison[~comparison.scenario.str.contains("-")].general_failure).all())
for k in ("prior_failing_scenarios", "prior_failures_reproduced_by_general", "single_event_verdicts_agree"):
    print(f"{k}: {summary[k]}")

,event,label,prior_failure,general_failure,populations_failing
0,underground_alarm,UA,False,False,"students, managers, bank tellers, drivers, stu..."
1,wildcat_strike,WS,False,False,
2,flash_mob,FM,False,False,bank tellers + drivers
3,long_rides,LR,False,False,"drivers, students + drivers, managers + driver..."
4,boycott_tncs,BT,False,False,"drivers, bank tellers + drivers"
5,underground_alarm-long_rides,UA + LR,True,True,"general, students, managers, bank tellers, dri..."
6,flash_mob-long_rides,FM + LR,True,False,"bank tellers, managers + drivers, bank tellers..."
7,flash_mob-boycott_tncs,FM + BT,True,False,"students, managers, managers + drivers"
8,wildcat_strike-long_rides,WS + LR,True,False,"students + drivers, bank tellers + drivers"
9,wildcat_strike-boycott_tncs,WS + BT,True,False,"students, managers, bank tellers, bank tellers..."


prior_failing_scenarios: ['UA + LR', 'FM + LR', 'FM + BT', 'WS + LR', 'WS + BT']
prior_failures_reproduced_by_general: ['UA + LR']
single_event_verdicts_agree: True


In [14]:
with open(os.path.join(outRoot, "rq1_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)
print(f"Wrote {os.path.join(outRoot, 'rq1_summary.json')}")

Wrote analysis_out/rq1-healthiness/rq1_summary.json
